In [2]:
!pip install torch


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Simple demo of a custom pytorch dataset and dataloader

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

In [5]:
class SimpleDataSet(Dataset):
    """
    A toy data set that returns feature and label pair
    feature = [index]
    label = [index * 2]
    """
    def __init__(self, size:int = 10):
        self.size = size

    def __len__(self) -> int:
        # Required method
        # gives the size of the dataset to the dataloader
        return self.size

    def __getitem__(self, idx: int):

        if idx >= self.size or idx < 0:
            raise IndexError("invalid index", idx)

        # create fake data based on the idx
        # torch tensors expect iterables as inputs, features are generally floats
        feature = torch.tensor([float(idx)], dtype=torch.float32)
        label = torch.tensor([float(idx * 2)], dtype=torch.float32)

        return feature, label      
  
    
    

In [6]:
dataset = SimpleDataSet(5)

loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0)

for batch_x, batch_y in loader:
    print("label batch: ", batch_x.tolist())
    print("feature batch: ", batch_y.tolist())
    print("---")

label batch:  [[3.0], [0.0]]
feature batch:  [[6.0], [0.0]]
---
label batch:  [[1.0], [4.0]]
feature batch:  [[2.0], [8.0]]
---
label batch:  [[2.0]]
feature batch:  [[4.0]]
---


## Converting variable length arrays to tensors of uniform length

variable-length multi-series records being stacked into uniform, zero-padded tensors.

In [8]:
from torch.nn.utils.rnn import pad_sequence

In [9]:
records = [
    {
        "series_a": [1.0, 2.0, 3.0],
        "series_b": [10.0, 20.0]
    },
    {
        "series_a": [4.0, 5.0],
        "series_b": [30.0, 40.0, 50.0, 60.0]
    },     
    {
        "series_a": [7.0, 8.0, 9.0],
        "series_b": [70.0, 90.0]
    }, 
]

In [10]:
series_a_list = [torch.tensor(r["series_a"], dtype=torch.float32) for r in records]
series_b_list = [torch.tensor(r["series_b"], dtype=torch.float32) for r in records]

In [16]:
series_a_padded = pad_sequence(series_a_list, batch_first=True, padding_value=0.0)
series_b_padded = pad_sequence(series_b_list, batch_first=True, padding_value=0.0)

In [17]:
print("series a padded's shape", series_a_padded.shape)
print("series a padded :\n ", series_a_padded)

series a padded's shape torch.Size([3, 3])
series a padded :
  tensor([[1., 2., 3.],
        [4., 5., 0.],
        [7., 8., 9.]])


In [18]:
print("series b padded's shape", series_b_padded.shape)
print("series b padded :\n ", series_b_padded)

series b padded's shape torch.Size([3, 4])
series b padded :
  tensor([[10., 20.,  0.,  0.],
        [30., 40., 50., 60.],
        [70., 90.,  0.,  0.]])


#### *Note*: The main point of converting a list into a tensor is to move your data into the numerical format that PyTorch (and GPUs) are designed to work with efficiently

## Categorical Feature Encoding

#### Mapping string descriptors (like "TCP", "UDP") into integer feature vectors or one hot vectors suitable for a neural network

In [21]:
labels = ["ASIA","NA", "EUR", "AUS", "EUR", "ASIA", "SA", "AUS"]
zones = sorted(set(labels))
loc_to_id = {loc:idx for idx, loc in enumerate(zones)}
encoded_labels = [loc_to_id[label] for label in labels]

In [26]:
num_classes = len(zones)
# the below line creates a matrix of zeroes
one_hot_tensor = torch.zeros(len(encoded_labels), len(zones), dtype=torch.float32)
"""
This line uses advanced encoding - It is the same as
one_hot_tensor[0, encoded_labels[0]] = 1.0
one_hot_tensor[1, encoded_labels[1]] = 1.0
one_hot_tensor[2, encoded_labels[2]] = 1.0
...
one_hot_tensor[N-1, encoded_labels[N-1]] = 1.0
"""
one_hot_tensor[range(len(encoded_labels)), encoded_labels] = 1.0

In [27]:
print("one_hot_tensor shape:", one_hot_tensor.shape)
print("one_hot_tensor:\n", one_hot_tensor)

one_hot_tensor shape: torch.Size([8, 5])
one_hot_tensor:
 tensor([[1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0.]])
